# RMSX/FlipBook Molstar Notebook Demo

This notebook shows two notebook-native paths for the RMSX Molstar FlipBook viewer:

1. Display existing PDB snapshots whose B-factors carry RMSX-like values.
2. Run RMSX on topology/trajectory inputs, then display the generated FlipBook snapshots inline.

The viewer is the same `rmsx-molstar-viewer/v1` Molstar experience used by the Galaxy prototype, but it runs directly in the notebook output instead of launching ChimeraX, VMD, or Galaxy.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd()
if not (PROJECT / "tools" / "rmsx").exists() and (PROJECT.parent / "tools" / "rmsx").exists():
    PROJECT = PROJECT.parent
TOOLS = PROJECT / "tools" / "rmsx"
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))

import importlib
import rmsx_molstar_notebook
importlib.reload(rmsx_molstar_notebook)
from rmsx_molstar_notebook import run_rmsx_notebook, view_flipbook_snapshots

OUTPUT = PROJECT / "notebook_outputs"
OUTPUT.mkdir(exist_ok=True)
OUTPUT

## Snapshot-only viewer

This cell creates a tiny synthetic snapshot set so the notebook viewer can be tested even before RMSX is installed. For real FlipBook output, replace `snapshot_dir` with a directory containing RMSX/FlipBook PDB snapshots such as `slice_1_first_frame.pdb`, `slice_2_first_frame.pdb`, and so on.

In [ ]:
def atom_line(serial, name, residue_id, x, y, z, bfactor):
    return (
        f"ATOM  {serial:5d} {name:^4s} ALA A{residue_id:4d}    "
        f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00{bfactor:6.2f}           {name[0]:>2s}"
    )

def make_snapshot(phase, stretch, offset):
    lines = []
    serial = 1
    for residue_id in range(1, 25):
        x = 4.5 * __import__("math").sin(residue_id * 0.35 + phase)
        y = residue_id * stretch
        z = 3.5 * __import__("math").cos(residue_id * 0.29 + phase)
        bfactor = offset + residue_id * 0.06
        lines.append(atom_line(serial, "N", residue_id, x - 0.8, y - 0.25, z, bfactor * 0.9)); serial += 1
        lines.append(atom_line(serial, "CA", residue_id, x, y, z + 0.2, bfactor)); serial += 1
        lines.append(atom_line(serial, "C", residue_id, x + 0.8, y + 0.25, z, bfactor * 0.95)); serial += 1
        lines.append(atom_line(serial, "O", residue_id, x + 1.1, y + 0.55, z - 0.3, bfactor * 0.95)); serial += 1
    lines.append("END")
    return "\n".join(lines) + "\n"

snapshot_dir = OUTPUT / "synthetic_snapshots"
snapshot_dir.mkdir(exist_ok=True)
for index, (phase, stretch, offset) in enumerate([(0.0, 0.95, 0.15), (0.8, 1.25, 0.45), (1.6, 1.55, 0.75)], start=1):
    (snapshot_dir / f"slice_{index}_first_frame.pdb").write_text(make_snapshot(phase, stretch, offset))

snapshot_result = view_flipbook_snapshots(
    snapshot_dir,
    palette="turbo",
    height=1040,
    write_html=OUTPUT / "synthetic_molstar_flipbook.html",
)
snapshot_result

## Full RMSX run

This cell runs RMSX against the bundled 1UBQ/mon_sys example and displays the generated snapshots inline. The RMSX executable must be available on `PATH`, or you can pass `rmsx_executable=[sys.executable, "-m", "rmsx"]` if you installed RMSX into the active notebook kernel environment.

In [ ]:
topology = PROJECT / "tools/rmsx/test-data/1UBQ.pdb"
trajectory = PROJECT / "tools/rmsx/test-data/mon_sys.dcd"

rmsx_result = run_rmsx_notebook(
    topology,
    trajectory,
    chain="7",
    num_slices=3,
    output_dir=OUTPUT / "rmsx_1ubq",
    palette="viridis",
    height=1040,
    write_html=OUTPUT / "rmsx_1ubq_molstar_flipbook.html",
)
rmsx_result

## Optional static plot display

If your RMSX run or Galaxy export also includes static heatmap/triple PNG files, this cell displays them next to the interactive FlipBook for comparison.

In [ ]:
from IPython.display import Image, display

plot_candidates = sorted((OUTPUT / "rmsx_1ubq").rglob("*.png"))
if not plot_candidates:
    print("No static RMSX PNG plots found yet. This is expected when RMSX was run with the notebook helper's default no-plot compute path.")
for plot in plot_candidates:
    print(plot)
    display(Image(filename=str(plot)))